# Chapter 6: Command-Line Data Manipulation

## 1. Introduction

Welcome to the core of command-line data mastery. In precision health, data is constantly in motion—flowing from sequencers to servers, from patient logs to alert systems. This section introduces pipes (`|`) and redirection (`>`), the fundamental tools for directing these data streams. You will learn how to save analysis results to files, chain commands into powerful, multi-step workflows, and cleanly separate useful output from error messages. Mastering these skills is the critical step toward building the automated, repeatable, and robust data pipelines essential for modern bioinformatics and clinical informatics.

---

## 2. Key Concepts and Definitions

*   **Standard Input (`stdin`)**: The data stream that feeds data *into* a command. By default, this is your keyboard, but it can be redirected to come from a file or another command's output. Medically, this could be a stream of real-time vital signs being fed to an analysis script.
*   **Standard Output (`stdout`)**: The data stream for a command's normal, expected results. When you `grep` a log file for "CRITICAL" events, the matching lines are sent to `stdout`. By default, this is displayed on your screen.
*   **Standard Error (`stderr`)**: A separate stream reserved for error messages and diagnostics. If a command fails (e.g., "file not found"), the error is sent to `stderr`. This keeps error messages from contaminating your results in `stdout`.
*   **Redirection**: The process of changing the destination of a data stream. This is like diverting a patient's lab results from the monitor screen directly into their electronic health record (EHR) file.
*   **Pipe (`|`)**: An operator that connects the `stdout` of one command directly to the `stdin` of another. This creates a "pipeline," allowing data to flow through multiple processing steps without creating intermediate files, much like an automated lab workflow where a sample moves directly from a centrifuge to an analyzer.
*   **Hypoglycemia**: A medical condition characterized by abnormally low blood glucose (blood sugar) levels, which can be a critical event requiring immediate attention.
*   **Tachycardia**: A heart rate that exceeds the normal resting rate (generally over 100 beats per minute in adults), often monitored in clinical event logs.
*   **STAT**: A standard medical term from the Latin *statim* ("immediately"), used to designate a task or order (like a lab test) that must be fulfilled with utmost urgency.
*   **Phenotype**: The set of observable characteristics of an organism, such as its morphology, development, and behavior. In medicine, this often refers to the clinical presentation of a disease.

---

## 3. Main Content

### 3.1 Saving Command Output (`>` and `>>`)

You can save the `stdout` of any command to a file using redirection operators. This is essential for preserving analysis results.

*   `>` **(Overwrite):** Creates a new file or completely overwrites an existing one with the output.
*   `>>` **(Append):** Adds the output to the end of an existing file, preserving its previous contents.

In [ ]:
%%bash
# Overwrite a file with new output from a vitals log
grep "Hypoglycemia" vitals_alerts.log > critical_alerts.txt
# Append more data to the same file
grep "Tachycardia" vitals_alerts.log >> critical_alerts.txt

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




> *Note: In a real clinical system, you might search for a specific event code or a structured tag like `SEVERITY:CRITICAL` for more precise filtering, rather than just a single word.*

### 3.2 Redirecting Input and Errors (`<`, `2>`, `&>`)

Beyond saving output, you can also provide input from files and manage error messages separately. This is vital for logging and debugging.

*   `<`: Redirects `stdin`, feeding a file's content to a command.
*   `2>`: Redirects `stderr` (stream #2), capturing only error messages.
*   `&>`: Redirects both `stdout` and `stderr` to the same destination.

In [ ]:
%%bash
# Redirect stdin from a file to the sort command
sort < gene_list.txt > gene_list_sorted.txt
# Redirect stdout and stderr to separate files
cat patient_data.csv non_existent_file.csv > full_output.log 2> error_log.txt
# Redirect both stdout and stderr to the same file
cat patient_data.csv non_existent_file.csv &> full_run.log

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




> **Pro Tip: Prefer Arguments Over Input Redirection**
> While `<` is useful for commands that *only* accept standard input, most modern tools (like `sort`, `grep`, `cat`) are designed to read files given as arguments (e.g., `sort gene_list.txt`). Using arguments is generally clearer and more readable than using input redirection.

### 3.3 Creating Command Pipelines (`|`)

The pipe (`|`) connects one command's `stdout` to another's `stdin`, allowing you to chain multiple analysis steps into a single, efficient command.

In [ ]:
%%bash
# Find, sort, and count unique critical entries
grep "CRITICAL" patient_vitals.csv | sort | uniq > critical_unique.log

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




This pipeline works in stages: `grep` finds the lines, `sort` arranges them, and `uniq` removes duplicates before the final result is saved.

> **Important: `sort` Before `uniq`**
> The `uniq` command only removes duplicate lines that are *adjacent* to each other. Always pipe your data through `sort` first to group all identical lines together. Failing to do so will result in an incomplete and inaccurate unique list, which could have serious consequences in data analysis.

### 3.4 Splitting Output with `tee`

Use `tee` to send `stdout` to a file while also passing it to the next command in a pipeline. This is perfect for saving intermediate results for auditing or debugging. By default, `tee` overwrites the file. Use the `-a` flag to append instead.

In [ ]:
%%bash
# Save urgent orders to a file AND count them
grep "STAT" pending_orders.txt | tee urgent_orders.log | wc -l
# Expected terminal output (example):
# 5
# Append abnormal results to a log AND count them
grep "ABNORMAL" lab_results.csv | tee -a master_abnormal.log | wc -l

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




> **In Practice: Creating Audit Trails with `tee`**
> In a clinical setting, you might build a script that processes patient admission data. Using `tee`, you can save a raw, intermediate list of all admissions processed on a given day to an audit file *while* the rest of the pipeline continues to filter for high-risk patients. This creates a permanent record for verification without interrupting the main workflow.

### 3.5 Quick Reference Table

| Operator | Name | Action | Example Use Case |
| :--- | :--- | :--- | :--- |
| `>` | Overwrite | Redirects `stdout` to a file, overwriting it. | `ls -l > file_list.txt` |
| `>>` | Append | Appends `stdout` to the end of a file. | `grep "Fever" log >> alerts.txt` |
| `<` | Input | Takes input for a command from a file. | `wc -l < gene_list.txt` |
| `|` | Pipe | Sends one command's `stdout` to another's `stdin`. | `grep "STAT" orders.txt | wc -l` |
| `tee` | Tee | Sends `stdout` to a file AND to the next command. | `grep "STAT" ... | tee stat.log | wc -l` |
| `2>` | Stderr Redirect | Redirects only error messages (`stderr`) to a file. | `run_analysis.sh 2> errors.log` |
| `&>` | Combined Redirect | Redirects both `stdout` and `stderr` to a file. | `run_analysis.sh &> full.log` |
| `uniq` | Unique | Filters adjacent, duplicate lines from input. | `sort genes.txt | uniq` |

---

## 4. Practice Exercises

### Exercise 1: Create a Unique Gene List (Basic)

**Objective:** Use a pipeline to process a file and save a clean version.
**Time:** 3 minutes
**Medical Context:** You have a raw list of gene identifiers from a sequencing run that contains many duplicates. To prepare for downstream analysis, you need a unique, sorted list.

Sort `gene_list.txt`, remove any duplicate lines, and save the result to `gene_list_unique.txt`.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
sort gene_list.txt | uniq > gene_list_unique.txt
```
**Explanation:** This command first sorts the contents of `gene_list.txt` alphabetically. The sorted output is then piped to `uniq`, which removes any adjacent duplicate lines. Finally, the `>` operator redirects the clean output to the new file.
**Key Learning:** Combining `sort` and `uniq` in a pipeline is the standard method for deduplicating data in a file.


</div>
</details>

### Exercise 2: Isolate Latest Critical Events (Intermediate)

**Objective:** Combine filtering and selection commands in a pipeline.
**Time:** 3 minutes
**Medical Context:** A clinician needs a quick summary of the most recent critical events from a continuous patient monitoring log that contains thousands of entries.

From `patient_vitals.csv`, find all lines containing "CRITICAL", and of that output, save only the last 10 lines to a new file named `latest_critical.log`.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
grep "CRITICAL" patient_vitals.csv | tail -n 10 > latest_critical.log
```
**Explanation:** `grep` first filters `patient_vitals.csv` to find all lines with the word "CRITICAL". This output is piped to `tail -n 10`, which selects only the last 10 lines from the data it receives. This final, small subset is then saved.
**Key Learning:** Pipelines allow you to filter a large dataset and then select a specific subset of the results in a single command.


</div>
</details>

### Exercise 3: Log and Count Abnormal Results (Intermediate)

**Objective:** Use `tee` to both save results and perform a subsequent action.
**Time:** 4 minutes
**Medical Context:** For auditing purposes, you must append all new abnormal lab results to a running log file while also getting a quick count of how many new results were found in the latest batch.

Find all "ABNORMAL" lines in `lab_results.csv`. Using a single pipeline, append these lines to `abnormal_events.log` and also display the total count on the terminal.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
grep "ABNORMAL" lab_results.csv | tee -a abnormal_events.log | wc -l
```
**Explanation:** `grep` finds the "ABNORMAL" lines. These are piped to `tee -a`, where the `-a` flag tells `tee` to append. The data is written to `abnormal_events.log` and simultaneously passed to `wc -l`, which counts the lines and prints the number to the screen.
**Key Learning:** `tee -a` is the perfect tool for logging intermediate data in a pipeline while allowing the data to continue to the next processing step.


</div>
</details>

> **Reflection Moment:** In a real-world scenario, why is it valuable to both append events to a master log (`abnormal_events.log`) and get an immediate count on the terminal? Consider the different needs of automated logging versus an on-duty analyst.

### Exercise 4: Separate Successful and Failed File Access (Basic)

**Objective:** Use `>` and `2>` to redirect `stdout` and `stderr` to different files.
**Time:** 3 minutes
**Medical Context:** An automated script needs to process a list of files. You must log which files were processed successfully and which ones failed to open, perhaps due to typos or permission issues.

Attempt to view `gene_list.txt` and a non-existent file `phenotypes.txt` using `cat`. Send valid output to `gene_list_contents.txt` and errors to `access_errors.log`.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
cat gene_list.txt phenotypes.txt > gene_list_contents.txt 2> access_errors.log
```
**Explanation:** The `cat` command successfully outputs the contents of `gene_list.txt` to `stdout`, which is redirected by `>` to `gene_list_contents.txt`. It then fails to find `phenotypes.txt` and generates an error message on `stderr`. This error stream is redirected by `2>` to `access_errors.log`.
**Key Learning:** Separating `stdout` and `stderr` is crucial for robust scripting and error handling.


</div>
</details>

### Exercise 5: Prepare a Sorted Critical Event Report (Intermediate)

**Objective:** Build a three-stage pipeline to filter, sort, and save data.
**Time:** 3 minutes
**Medical Context:** To prepare a report for a morning briefing, you must extract all "CRITICAL" events from `patient_vitals.csv`, sort them alphabetically for clarity, and save the result to `critical_sorted.log`.

Create a single command pipeline to accomplish this.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
grep "CRITICAL" patient_vitals.csv | sort > critical_sorted.log
```
**Explanation:** `grep` first extracts all lines containing "CRITICAL" from the vitals file. This filtered output is then piped directly to the `sort` command, which arranges the lines alphabetically. Finally, the sorted list of critical events is redirected into the report file `critical_sorted.log`.
**Key Learning:** Complex data preparation tasks can often be accomplished by chaining simple, single-purpose tools together with pipes.


</div>
</details>

---

## 5. Practical Applications

*   **Genomic Variant Filtering:** A researcher can use a pipeline like `grep "HIGH_IMPACT" variants.vcf | sort -k1,1 -k2,2n | uniq > high_impact_variants.txt` to process a large Variant Call Format (VCF) file. This technique extracts only high-impact variants, sorts them by chromosome and position, removes duplicates, and saves the clean list for investigating disease-causing mutations.
*   **Real-time Clinical Alerting:** A hospital script could use `tail -f /var/log/patient_monitor.log | grep --line-buffered "SepsisAlert"` to continuously watch a log file for sepsis alerts. The output could be piped to another script that sends a page to a clinician, demonstrating a real-time alerting pipeline built with simple command-line tools.
*   **Pharmacogenomics Data Aggregation:** An analyst can combine data from multiple patient cohorts using a command like `cat cohort*.csv | grep "Warfarin" | tee warfarin_patients.log | wc -l`. This technique uses `cat` to merge files, `grep` to filter for a specific drug, `tee` to save the combined data for deep analysis, and `wc -l` to provide a quick count of the total patients involved.

---

## 6. Summary and Key Takeaways

In this section, we've explored how to control and connect command-line data streams using pipes and redirection. These tools are the foundation of powerful, automated data processing in bioinformatics and precision health. By mastering them, you can build complex workflows that are both efficient and easy to manage.

*   **Three Core Streams:** All commands operate on `stdin` (input), `stdout` (output), and `stderr` (errors). Understanding this separation is key.
*   **Redirection Provides Control:** Use `>` and `>>` to save results to files, and `2>` to capture errors separately, which is critical for debugging automated scripts.
*   **Pipes Build Workflows:** The pipe `|` operator is the glue that connects individual commands, allowing you to filter, sort, and process data in a single, elegant line.
*   **`tee` Creates Audit Points:** Use `tee` when you need to save data at a midpoint in a pipeline while also passing it along for further processing.

With this knowledge, you are now prepared to move from running single commands to constructing sophisticated shell scripts that can automate complex, multi-step tasks.

---


---

## 📝 Interactive Practice

Practice the concepts with these interactive exercises:

### *   `>>` **(Append):** Adds the output to the end of an existing file, preserving its previous contents.

In [ ]:
%%bash
# Overwrite a file with new output from a vitals log
grep "Hypoglycemia" vitals_alerts.log > critical_alerts.txt
# Append more data to the same file
grep "Tachycardia" vitals_alerts.log >> critical_alerts.txt

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### *   `&>`: Redirects both `stdout` and `stderr` to the same destination.

In [ ]:
%%bash
# Redirect stdin from a file to the sort command
sort < gene_list.txt > gene_list_sorted.txt
# Redirect stdout and stderr to separate files
cat patient_data.csv non_existent_file.csv > full_output.log 2> error_log.txt
# Redirect both stdout and stderr to the same file
cat patient_data.csv non_existent_file.csv &> full_run.log

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### The pipe (`|`) connects one command's `stdout` to another's `stdin`, allowing you to chain multiple analysis steps into a single, efficient command.

In [ ]:
%%bash
# Find, sort, and count unique critical entries
grep "CRITICAL" patient_vitals.csv | sort | uniq > critical_unique.log

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### o passing it to the next command in a pipeline. This is perfect for saving intermediate results for auditing or debugging. By default, `tee` overwrites the file. Use the `-a` flag to append instead.

In [ ]:
%%bash
# Save urgent orders to a file AND count them
grep "STAT" pending_orders.txt | tee urgent_orders.log | wc -l
# Expected terminal output (example):
# 5
# Append abnormal results to a log AND count them
grep "ABNORMAL" lab_results.csv | tee -a master_abnormal.log | wc -l

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
sort gene_list.txt | uniq > gene_list_unique.txt

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
grep "CRITICAL" patient_vitals.csv | tail -n 10 > latest_critical.log

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
grep "ABNORMAL" lab_results.csv | tee -a abnormal_events.log | wc -l

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
cat gene_list.txt phenotypes.txt > gene_list_contents.txt 2> access_errors.log

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
grep "CRITICAL" patient_vitals.csv | sort > critical_sorted.log

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above


